# Check 2 -- do weak rules get in, and should every method get a precision gate?

Paper mini-experiment. Logic: `thesis.system_eval.rule_precision_gate` (gate +
rule log) and the shared walk-forward loop in `signature_leakage.py`; run with
`shell-scripts/paper/run_rule_precision_gate.sh` (DGX).

## Step 1 -- what the code does

- **Counts are raw.** `extract_leaf_rules` counts each leaf's groups with
  `tree.apply(X)`, not the class-weighted `tree_.value`.
- **Polarity is a class-balanced majority.** A leaf is attack-leaning iff
  `confidence_attack > confidence_benign`, and both are per-class **recall**
  (n_class_in_leaf / n_class_total). That is exactly *attack rate in the leaf
  > π* (growth rate > 1): a 3-in-100 leaf at π ≈ 1.5% is "attack". The thesis
  text ("whichever class is more common decides") does not match the code.
- **Weak rules can only enter through round-1 tree leaves.** Contrast items
  need growth ≥ 3, diverse-round leaves (round ≥ 2) need
  `round_min_growth_rate` ≥ 3; round 1 has no floor.
- **Stored confidence is recall** (mined features, `SymbolicFeature`,
  `DynamicSchema.support_*`). The drift monitor's *mined* rule confidence was
  `max(recall_attack, recall_benign)` while signal 2 *observes* precision --
  fixed in `features/dynamic_schema_builder.py` (now: leaf precision in its
  predicted class). Experiment 4's signal-2 results predate the fix.

## The gate

Attack-leaning: attack rate in the rule ≥ g·π; benign-leaning: ≤ π/g; both on
the window the rule was mined from. Main g = 2, sensitivity g ∈ {1.5, 3}.
Growth 3 ↔ attack rate ≈ 4.4% < 3π, so at g = 3 borderline contrast items fail
too, not only weak leaves.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from thesis.paths import RESULTS_DIR, ROOT
from thesis.system_eval import signature_leakage as sl
from thesis.system_eval.rule_precision_gate import GATE_GS, MAIN_G

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)


def to_booktabs(df: pd.DataFrame, caption: str, label: str, float_fmt: str = "{:.3f}") -> str:
    styler = df.style.hide(axis="index").format(
        {c: float_fmt for c in df.select_dtypes("float").columns}
    ).format_index(escape="latex", axis=1).format(escape="latex-math", subset=df.select_dtypes("object").columns)
    return styler.to_latex(hrules=True, caption=caption, label=label, position="t", position_float="centering")

## Load

In [ ]:
RUN_DIR = None  # None -> latest full run (results mirror), else a path

def _latest_run() -> Path:
    for base in (RESULTS_DIR / "paper" / "rule-precision-gate",
                 ROOT / "artifacts" / "experiments" / "rule_precision_gate"):
        runs = sorted(d for d in base.glob("*") if (d / "ablation_summary.csv").exists()) if base.exists() else []
        if runs:
            return runs[-1]
    raise FileNotFoundError("no rule_precision_gate run found")

run_dir = Path(RUN_DIR) if RUN_DIR else _latest_run()
per_step = pd.read_csv(run_dir / "per_step_results.csv")
rules = pd.read_csv(run_dir / "rules.csv")
print(run_dir, "|", per_step.groupby("arm").size().to_dict(), "rows per arm |", len(rules), "rule rows")

## §2 Distribution of attack rate inside the rules (mining window)

One point per (step, rule). Zero rates are drawn at the floor of the axis.
Dashed: π (median over steps); dotted: g·π and π/g for g = 2.

In [ ]:
pi = rules.groupby("step_i")["base_rate"].first().median()
FLOOR = 1e-4
fig, ax = plt.subplots(figsize=(8, 4))
rng = np.random.default_rng(0)
for k, (lab, d) in enumerate(rules.groupby("source_label")):
    for j, (mt, dd) in enumerate(d.groupby("mining_type")):
        x = k + (j - 0.5) * 0.3 + rng.uniform(-0.08, 0.08, len(dd))
        ax.scatter(x, dd["attack_rate"].clip(lower=FLOOR), s=8, alpha=0.5, label=f"{lab} / {mt}")
ax.axhline(pi, color="k", ls="--", lw=1, label=f"π = {pi:.4f}")
ax.axhline(MAIN_G * pi, color="tab:red", ls=":", lw=1, label=f"{MAIN_G:g}·π")
ax.axhline(pi / MAIN_G, color="tab:blue", ls=":", lw=1, label=f"π/{MAIN_G:g}")
ax.set_yscale("log"); ax.set_ylabel("attack rate in rule (Wᵢ)")
ax.set_xticks(range(rules["source_label"].nunique()), sorted(rules["source_label"].unique()))
ax.legend(fontsize=7, ncol=2); plt.tight_layout()

### Share of rules failing the gate, per leaning class and mining type

`never_fires` = the rule's encoded column is all-zero on its own mining window (a dead feature, see the contrast-item encoding issue), counted inside the fail shares.

In [ ]:
fail = rules.assign(**{f"fails_g{g:g}": ~rules[f"passes_g{g:g}"].astype(bool) for g in GATE_GS})
fail_tab = (fail.groupby(["source_label", "mining_type"])
            [[f"fails_g{g:g}" for g in GATE_GS]].mean()
            .join(fail.groupby(["source_label", "mining_type"]).size().rename("n_rules"))
            .join(fail.assign(never_fires=fail["n_fire"] == 0)
                  .groupby(["source_label", "mining_type"])["never_fires"].mean()))
# never_fires: the rule's column is constant 0 on its own mining window -- a
# dead feature, not a weak one. Report it separately from "fails the gate".
fail_tab

In [ ]:
print("share of ALL rules failing:", {g: round(float((~rules[f'passes_g{g:g}'].astype(bool)).mean()), 3) for g in GATE_GS})
weak_attack = rules[(rules["source_label"] == "attack") & (rules["attack_rate"] < 0.5)]
print(f"attack-leaning rules that are mostly benign (attack rate < 50%): {len(weak_attack)} / "
      f"{(rules['source_label'] == 'attack').sum()}  (min attack rate {rules.loc[rules['source_label']=='attack','attack_rate'].min():.4f})")

## §3 Downstream: gate off vs. on

In [ ]:
summary = sl.summarize_ablation(per_step)
feat = per_step.groupby("arm")[["n_features", "n_symbolic"]].mean()
summary = summary.merge(feat, left_on="arm", right_index=True)
summary[["arm", "arm_label", "f1_mean", "f1_std", "pr_auc_mean", "pr_auc_std",
         "fp_per_day_mean", "fp_per_day_std", "n_features", "n_symbolic"]]

In [ ]:
def pm(df, metric, d=3):
    return df[f"{metric}_mean"].map(f"{{:.{d}f}}".format) + " ± " + df[f"{metric}_std"].map(f"{{:.{d}f}}".format)

paper = pd.DataFrame({
    "Setting": summary["arm_label"],
    "F1": pm(summary, "f1"), "PR-AUC": pm(summary, "pr_auc"), "FP/day": pm(summary, "fp_per_day", 1),
    "#features": summary["n_features"].map("{:.1f}".format),
})
display(paper)
print(to_booktabs(paper.apply(lambda c: c.str.replace("±", r"$\pm$", regex=False)),
      "Precision gate on mined rules (RF, 5 seeds, $\\gamma=0.05$): mean $\\pm$ std over seeds; "
      "\\#features averaged over windows.", "tab:check2-gate"))

### Helps / hurts per window

Per step, mean F1 over seeds of each gated arm minus gate-off; a window counts
as *helps* / *hurts* if the difference exceeds ±0.005, else *same*.

In [ ]:
TOL = 0.005
f1 = per_step.groupby(["arm", "step_i"])["f1"].mean().unstack(0)
rows = []
for arm in [a for a in f1.columns if a != "b"]:
    d = f1[arm] - f1["b"]
    rows.append({"arm": arm, "helps": int((d > TOL).sum()), "same": int((d.abs() <= TOL).sum()),
                 "hurts": int((d < -TOL).sum()), "mean_delta_f1": d.mean()})
helps = pd.DataFrame(rows)
helps

## §4 Stability on the next window (bonus)

For rules that pass the gate at g = 2: how often is their attack rate on Wᵢ₊₁ on
the same side of that window's π as on Wᵢ? (Rules that don't fire on Wᵢ₊₁ are
left out.) The monitor's calibration signal watches exactly this.

In [ ]:
r = rules.dropna(subset=["same_side_next"])
stab = (r.groupby([r[f"passes_g{MAIN_G:g}"].astype(bool).rename("passes_g2"), "source_label"])["same_side_next"]
        .agg(["mean", "size"]).rename(columns={"mean": "share_same_side", "size": "n"}))
stab

## Decision

- Gate removes features and F1 stays the same or improves → turn it on (g = 2)
  for every method in the comparison.
- Gate clearly hurts F1 → keep it off, report the distribution, and soften
  "each feature has a monitorable precision".

In [ ]:
s = summary.set_index("arm")
if "b" in s.index and f"b_g{MAIN_G:g}" in s.index:
    off, on = s.loc["b"], s.loc[f"b_g{MAIN_G:g}"]
    d = on["f1_mean"] - off["f1_mean"]
    print(f"F1 gate off {off['f1_mean']:.4f} ± {off['f1_std']:.4f} | g={MAIN_G:g} {on['f1_mean']:.4f} ± {on['f1_std']:.4f} | Δ={d:+.4f}")
    print(f"features off {off['n_features']:.1f} -> on {on['n_features']:.1f}")
    stds = [v for v in (off["f1_std"], on["f1_std"]) if np.isfinite(v)]
    noise = max(stds) if stds else 0.0  # single seed: no noise estimate
    print("verdict:", "gate on (no loss)" if d >= -noise else "gate hurts F1 -- keep off, soften the claim")